# Generate mock analytics data

This notebook creates synthetic usage data for the business dashboard.

The data represents simulated parent searches and venue activity. It is intended only for demonstrating the MVP and does not represent real users or confirmed customer behaviour.

In [ ]:
import pandas as pd
import random

from datetime import datetime, timedelta
from pathlib import Path

## Set a random seed

A fixed random seed ensures that the notebook generates the same demo data each time it is run.

In [ ]:
random.seed(42)

## Configure the mock dataset

The mock dataset will cover the previous 60 days.

The IDs below must correspond to records in the development database. The current seeded database contains parent users with IDs 2 and 3, five sample venues and ten amenities.

In [ ]:
number_of_searches = 300
number_of_days = 60

user_ids = [2, 3]
venue_ids = [1, 2, 3, 4, 5]
amenity_ids = list(range(1, 11))

categories = [
    "catering.cafe",
    "catering.restaurant",
    "entertainment.museum",
    "leisure.playground"
]

output_directory = Path("../mock")
output_directory.mkdir(parents=True, exist_ok=True)

## Generate mock search events

Each record represents a parent searching the app.

Searches are given a simulated user, date and time, location, category and number of results. The generated coordinates are located approximately around Hertfordshire.

In [ ]:
search_events = []

end_date = datetime.now()
start_date = end_date - timedelta(days=number_of_days)

for search_id in range(1, number_of_searches + 1):
    random_days = random.randint(0, number_of_days - 1)

    search_hour = random.choices(
        population=[8, 10, 12, 14, 16, 18],
        weights=[5, 15, 30, 25, 15, 10],
        k=1
    )[0]

    searched_at = (
        start_date
        + timedelta(days=random_days)
    ).replace(
        hour=search_hour,
        minute=random.randint(0, 59),
        second=0,
        microsecond=0
    )

    search_events.append({
        "id": search_id,
        "user_id": random.choice(user_ids),
        "latitude": round(random.uniform(51.70, 51.95), 6),
        "longitude": round(random.uniform(-0.45, 0.15), 6),
        "radius": random.choice([5000, 10000, 15000]),
        "category": random.choice(categories),
        "result_count": random.randint(5, 30),
        "searched_at": searched_at
    })

search_events_df = pd.DataFrame(search_events)

display(search_events_df.head())
print(f"{len(search_events_df)} mock search events created")

,id,user_id,latitude,longitude,radius,category,result_count,searched_at
0,1,3,51.761223,-0.366277,5000,catering.cafe,23,2026-09-05 10:47:00
1,2,2,51.758165,-0.088789,15000,catering.restaurant,27,2026-08-23 08:05:00
2,3,2,51.812302,-0.283086,5000,catering.restaurant,27,2026-09-06 14:26:00
3,4,2,51.939303,-0.248043,5000,leisure.playground,8,2026-08-23 12:09:00
4,5,3,51.901782,-0.012161,15000,catering.cafe,17,2026-08-18 16:38:00


300 mock search events created


## Review the mock search data

The date range and category counts are checked to ensure that the generated dataset is suitable for the dashboard.

In [ ]:
print("First date:", search_events_df["searched_at"].min())
print("Last date:", search_events_df["searched_at"].max())

display(
    search_events_df["category"].value_counts()
)

First date: 2026-07-27 10:09:00
Last date: 2026-09-24 14:44:00


category
leisure.playground      90
catering.restaurant     73
entertainment.museum    72
catering.cafe           65
Name: count, dtype: int64

## Generate requested amenities

Each search will include between one and three requested amenities.

The amenities are weighted so that family-focused facilities such as changing facilities, pram access and high chairs appear more frequently. This creates realistic patterns for the demonstration dashboard.

In [ ]:
amenity_names = {
    1: "Accessible entrance",
    2: "Accessible toilet",
    3: "Prams allowed",
    4: "Pram storage",
    5: "Changing facilities",
    6: "Table reservation",
    7: "Breastfeeding friendly",
    8: "Children's activities",
    9: "Parking",
    10: "High chairs"
}

amenity_weights = {
    1: 10,
    2: 14,
    3: 22,
    4: 12,
    5: 28,
    6: 8,
    7: 18,
    8: 16,
    9: 12,
    10: 20
}

## Link amenities to search events

Each selected amenity creates one record in `search_event_amenities`.

A set is used to prevent the same amenity from being added to the same search more than once.

In [ ]:
search_event_amenities = []

weighted_amenity_ids = []

for amenity_id, weight in amenity_weights.items():
    weighted_amenity_ids.extend([amenity_id] * weight)

for search_id in search_events_df["id"]:
    number_requested = random.choices(
        population=[1, 2, 3],
        weights=[50, 35, 15],
        k=1
    )[0]

    selected_amenities = set()

    while len(selected_amenities) < number_requested:
        selected_amenities.add(
            random.choice(weighted_amenity_ids)
        )

    for amenity_id in selected_amenities:
        search_event_amenities.append({
            "search_event_id": search_id,
            "amenity_id": amenity_id
        })

search_event_amenities_df = pd.DataFrame(
    search_event_amenities
)

display(search_event_amenities_df.head())

print(
    f"{len(search_event_amenities_df)} "
    "search-amenity records created"
)

,search_event_id,amenity_id
0,1,5
1,2,2
2,3,8
3,4,9
4,4,7


501 search-amenity records created


## Review amenity demand

The number of searches requesting each amenity is calculated.

These figures will support the “Most requested amenities” section of the business dashboard.

In [ ]:
amenity_demand = (
    search_event_amenities_df
    .groupby("amenity_id")
    .size()
    .reset_index(name="search_count")
)

amenity_demand["amenity_name"] = (
    amenity_demand["amenity_id"]
    .map(amenity_names)
)

amenity_demand = amenity_demand.sort_values(
    "search_count",
    ascending=False
)

display(
    amenity_demand[
        ["amenity_name", "search_count"]
    ]
)

,amenity_name,search_count
4,Changing facilities,86
6,Breastfeeding friendly,75
2,Prams allowed,71
9,High chairs,60
7,Children's activities,54
1,Accessible toilet,52
8,Parking,33
3,Pram storage,27
5,Table reservation,22
0,Accessible entrance,21


## Generate mock venue views

A venue view represents a parent opening a venue's details page.

Views are generated using the same users and date range as the search events. Some venues are given higher probabilities to simulate differences in popularity.

In [ ]:
number_of_views = 180

venue_popularity = {
    1: 30,
    2: 25,
    3: 20,
    4: 15,
    5: 10
}

weighted_venue_ids = []

for venue_id, weight in venue_popularity.items():
    weighted_venue_ids.extend([venue_id] * weight)

## Create the venue-view records

Each venue view is linked to a demo user and a venue.

The view time is based on a randomly selected search time, with a short delay added to represent the user opening a venue after searching.

In [ ]:
venue_views = []

for view_id in range(1, number_of_views + 1):
    related_search = search_events_df.sample(
        n=1,
        random_state=random.randint(1, 100000)
    ).iloc[0]

    viewed_at = (
        related_search["searched_at"]
        + timedelta(minutes=random.randint(1, 20))
    )

    venue_views.append({
        "id": view_id,
        "venue_id": random.choice(weighted_venue_ids),
        "user_id": related_search["user_id"],
        "viewed_at": viewed_at
    })

venue_views_df = pd.DataFrame(venue_views)

display(venue_views_df.head())

print(f"{len(venue_views_df)} mock venue views created")

,id,venue_id,user_id,viewed_at
0,1,1,2,2026-07-27 11:06:00
1,2,4,2,2026-09-22 18:48:00
2,3,5,2,2026-09-06 11:02:00
3,4,1,3,2026-09-22 12:57:00
4,5,4,2,2026-09-14 14:51:00


180 mock venue views created


## Review venue popularity

The number of views for each venue is counted.

These totals can be displayed in the business dashboard and used to compare engagement between venues.

In [ ]:
venue_view_counts = (
    venue_views_df
    .groupby("venue_id")
    .size()
    .reset_index(name="view_count")
    .sort_values("view_count", ascending=False)
)

display(venue_view_counts)

,venue_id,view_count
1,2,49
0,1,45
2,3,35
3,4,32
4,5,19


## Export the mock analytics data

The generated datasets are saved as CSV files so they can be reviewed and imported into the development database.

These files contain simulated data and must not be described as real customer activity.

In [ ]:
search_events_df.to_csv(
    output_directory / "search_events_mock.csv",
    index=False
)

search_event_amenities_df.to_csv(
    output_directory / "search_event_amenities_mock.csv",
    index=False
)

venue_views_df.to_csv(
    output_directory / "venue_views_mock.csv",
    index=False
)

print("Mock analytics files saved:")
print(output_directory / "search_events_mock.csv")
print(output_directory / "search_event_amenities_mock.csv")
print(output_directory / "venue_views_mock.csv")

Mock analytics files saved:
..\mock\search_events_mock.csv
..\mock\search_event_amenities_mock.csv
..\mock\venue_views_mock.csv
